<a href="https://colab.research.google.com/github/CostanzoPadovano/my_function/blob/main/ASTRONOMIA_OGGETTI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
pip install pandas astropy pytz

In [11]:
# Prima di eseguire, installa le librerie necessarie con questo comando nel terminale:
# pip install pandas astropy pytz numpy

import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import pytz
import warnings
warnings.filterwarnings('ignore')

# Importa le librerie astronomiche
from astropy.coordinates import EarthLocation, SkyCoord, AltAz, ICRS # Import ICRS
# from astropy.coordinates.solar_system import get_sun, get_moon # Modified import - Commented out
from astropy.time import Time
import astropy.units as u
from astroplan import Observer # Import already exists

# --- PARAMETRI DA PERSONALIZZARE ---
LATITUDE = 41.702426
LONGITUDE = 15.730919
ELEVATION = 566
START_HOUR = 20
START_MINUTE = 30
END_HOUR = 2
END_MINUTE = 0
MIN_ALTITUDE = 20
MAX_MAGNITUDE = 14.0  # Magnitudine limite per telescopi amatoriali
MIN_SIZE = 1.0  # Dimensione minima in arcominuti
TOP_Nebulose = 20
# --- FINE DEI PARAMETRI ---

def calculate_sky_brightness(moon_phase, moon_altitude):
    """
    Stima la qualità del cielo basata sulla fase lunare e altezza della Luna.
    """
    if moon_altitude < 0:
        return "Eccellente (Luna sotto l'orizzonte)"
    elif moon_phase < 0.25:
        return "Ottima (Luna crescente)"
    elif moon_phase < 0.5:
        return "Buona (Primo quarto)"
    elif moon_phase < 0.75:
        return "Discreta (Luna gibbosa)"
    else:
        return "Scarsa (Luna piena)"

def get_optimal_observation_time(coords, location, start_time, end_time):
    """
    Calcola il momentOptimal_Timeo ottimale di osservazione (massima altezza).
    """
    times = Time(np.linspace(start_time.jd, end_time.jd, 20), format='jd')
    altaz_frames = [AltAz(obstime=t, location=location) for t in times]
    altitudes = [coords.transform_to(frame).alt.degree for frame in altaz_frames]

    max_idx = np.argmax(altitudes)
    optimal_time = times[max_idx]
    max_altitude = altitudes[max_idx]

    return optimal_time, max_altitude # Return Time object


def calculate_airmass(altitude):
    """
    Calcola la massa d'aria per stimare l'estinzione atmosferica.
    """
    if altitude <= 0:
        return np.inf
    z = np.radians(90 - altitude)
    return 1 / np.cos(z) if altitude > 30 else 1 / (np.cos(z) + 0.025 * np.exp(-11 * np.cos(z)))

def estimate_surface_brightness(magnitude, size_maj, size_min):
    """
    Stima la brillanza superficiale dell'oggetto.
    """
    if pd.isna(magnitude) or pd.isna(size_maj):
        return np.nan

    # Usa size_min se disponibile, altrimenti assume oggetto circolare
    if pd.isna(size_min):
        size_min = size_maj

    # Calcola l'area in arcosecondi quadrati
    area_arcsec2 = np.pi * (size_maj * 30) * (size_min * 30)  # converti da arcmin a arcsec

    if area_arcsec2 > 0:
        # Formula per brillanza superficiale
        return magnitude + 2.5 * np.log10(area_arcsec2)
    return np.nan

def categorize_difficulty(magnitude, surface_brightness, size):
    """
    Categorizza la difficoltà di osservazione.
    """
    if pd.isna(magnitude):
        return "Dati insufficienti"

    if magnitude < 8:
        if not pd.isna(size) and size > 10:
            return "Facile (binocolo)"
        return "Facile"
    elif magnitude < 10:
        return "Moderata"
    elif magnitude < 12:
        if not pd.isna(surface_brightness) and surface_brightness > 22:
            return "Difficile"
        return "Impegnativa"
    else:
        return "Molto difficile"

def run_visibility_check():
    """
    Funzione principale che esegue il caricamento dei dati e i calcoli.
    """
    # 1. IMPOSTAZIONE DI LUOGO E ORA
    location = EarthLocation(lat=LATITUDE*u.deg, lon=LONGITUDE*u.deg, height=ELEVATION*u.m)
    tz = pytz.timezone('Europe/Rome')
    now = datetime.now(tz)
    start_time_local = now.replace(hour=START_HOUR, minute=START_MINUTE, second=0, microsecond=0)

    if END_HOUR < START_HOUR:
        end_time_local = (now + timedelta(days=1)).replace(hour=END_HOUR, minute=END_MINUTE, second=0, microsecond=0)
    else:
        end_time_local = now.replace(hour=END_HOUR, minute=END_MINUTE, second=0, microsecond=0)

    time_window_start = Time(start_time_local)
    time_window_end = Time(end_time_local)

    # Calcola il tempo medio per informazioni su Luna e Sole
    time_middle = Time((time_window_start.jd + time_window_end.jd) / 2, format='jd')

    print(f"🔭 PIANO DI OSSERVAZIONE NEBULOSE - {start_time_local.strftime('%d/%m/%Y')}")
    print(f"📍 Località: Lat {LATITUDE:.4f}°, Lon {LONGITUDE:.4f}°, Alt {ELEVATION}m")
    print(f"⏰ Finestra osservativa: {start_time_local.strftime('%H:%M')} - {end_time_local.strftime('%H:%M')}")
    print("=" * 80)

    # 2. CALCOLO CONDIZIONI ASTRONOMICHE
    observer = Observer(location=location, timezone=tz) # Create an Observer object

    # Use Observer to get Sun and Moon AltAz coordinates
    sun_altaz = observer.sun_altaz(time_middle)
    moon_altaz = observer.moon_altaz(time_middle)

    # Transform AltAz to ICRS to get RA/Dec for phase calculation
    sun_coord_icrs = sun_altaz.transform_to(ICRS())
    moon_coord_icrs = moon_altaz.transform_to(ICRS())


    # Calcola la fase lunare (approssimativa)
    sun_lon = sun_coord_icrs.ra.degree
    moon_lon = moon_coord_icrs.ra.degree
    phase_angle = (moon_lon - sun_lon) % 360
    moon_phase = phase_angle / 360

    print("\n🌙 CONDIZIONI DEL CIELO:")
    print(f"   Altezza del Sole a metà serata: {sun_altaz.alt.degree:.1f}°")
    print(f"   Altezza della Luna: {moon_altaz.alt.degree:.1f}°")
    print(f"   Fase lunare: {moon_phase*100:.0f}%")
    print(f"   Qualità del cielo: {calculate_sky_brightness(moon_phase, moon_altaz.alt.degree)}")
    print("=" * 80)

    # 3. CARICAMENTO E FILTRAGGIO DEL CATALOGO
    url = "https://raw.githubusercontent.com/mattiaverga/OpenNGC/refs/heads/master/database_files/NGC.csv"
    try:
        print("\n📂 Caricamento catalogo NGC...")
        # Use the 'python' engine and quotechar for robustness
        # Removed low_memory=False as it's not supported with engine='python'
        df = pd.read_csv(url, sep=';', on_bad_lines='skip', engine='python', quotechar='"')
        df.columns = df.columns.str.strip()

        # Conversione delle colonne numeriche
        numeric_columns = ['V-Mag', 'B-Mag', 'MajAx', 'MinAx']
        for col in numeric_columns:
            if col in df.columns:
                df[col] = pd.to_numeric(df[col], errors='coerce')

    except Exception as e:
        print(f"❌ Errore nel caricamento del file: {e}")
        return

    # Tipi di nebulose con descrizioni
    nebula_types = {
        'PN': 'Planetaria',
        'HII': 'Emissione HII',
        'Neb': 'Nebulosa',
        'SNR': 'Resto Supernova',
        'RN': 'Riflessione',
        'EmN': 'Emissione',
        'Cl+N': 'Ammasso+Nebulosa'
    }

    df_nebulae = df[df['Type'].isin(nebula_types.keys())].dropna(subset=['RA', 'Dec']).copy()

    # Applica filtri di magnitudine e dimensione se specificati
    if MAX_MAGNITUDE:
        df_nebulae = df_nebulae[(df_nebulae['V-Mag'] <= MAX_MAGNITUDE) | df_nebulae['V-Mag'].isna()]

    if MIN_SIZE:
        df_nebulae = df_nebulae[(df_nebulae['MajAx'] >= MIN_SIZE) | df_nebulae['MajAx'].isna()]

    print(f"✅ Trovate {len(df_nebulae)} nebulose nel catalogo dopo i filtri iniziali.")

    # 4. CALCOLO ASTRONOMICO AVANZATO
    print("🔬 Calcolo visibilità e parametri osservativi...")

    coords = SkyCoord(df_nebulae['RA'], df_nebulae['Dec'], unit=(u.hourangle, u.deg), frame='icrs')

    # Use observer to calculate altitudes over the time window
    times_for_alt = Time(np.linspace(time_window_start.jd, time_window_end.jd, 50), format='jd') # More points for better max alt calculation

    # Calculate altitudes for each object over the time window
    altitudes_list = []
    for coord in coords:
        altitudes_list.append(observer.altaz(times_for_alt, coord).alt.degree)

    altitudes_over_time = np.array(altitudes_list).T # Transpose to have shape (50, 260)

    df_nebulae['Alt_start'] = altitudes_over_time[0] # Altitude at the start of the window
    df_nebulae['Alt_end'] = altitudes_over_time[-1] # Altitude at the end of the window
    # Calculate Azimuth at the start for each object
    df_nebulae['Az_start'] = observer.altaz(time_window_start, coords).az.degree


    # Calculate time and height of maximum altitude within the window
    max_altitudes = np.max(altitudes_over_time, axis=0)
    max_alt_indices = np.argmax(altitudes_over_time, axis=0)
    optimal_times_utc = times_for_alt[max_alt_indices]

    df_nebulae['Max_Alt'] = max_altitudes

    # Convert optimal times to local timezone strings
    optimal_times_local = []
    for t_utc in optimal_times_utc:
        local_dt = pytz.UTC.localize(t_utc.to_datetime()).astimezone(tz)
        optimal_times_local.append(local_dt.strftime('%H:%M'))

    df_nebulae['Optimal_Time'] = optimal_times_local


    # Calcola massa d'aria minima
    df_nebulae['Min_Airmass'] = df_nebulae['Max_Alt'].apply(calculate_airmass)

    # Calcola brillanza superficiale
    df_nebulae['Surface_Brightness'] = df_nebulae.apply(
        lambda row: estimate_surface_brightness(row['V-Mag'], row['MajAx'], row['MinAx']),
        axis=1
    )

    # Categorizza difficoltà
    df_nebulae['Difficulty'] = df_nebulae.apply(
        lambda row: categorize_difficulty(row['V-Mag'], row['Surface_Brightness'], row['MajAx']),
        axis=1
    )

    # Aggiungi descrizione del tipo
    df_nebulae['Type_Desc'] = df_nebulae['Type'].map(nebula_types)

    # Calcola separazione dalla Luna at time_middle
    moon_coord_icrs = moon_altaz.transform_to(ICRS()) # Use the ICRS coordinate calculated earlier
    moon_sep = coords.separation(moon_coord_icrs)
    df_nebulae['Moon_Sep'] = moon_sep.degree


    # Filtra per visibilità
    is_visible_mask = (df_nebulae['Max_Alt'] > MIN_ALTITUDE)
    visible_df = df_nebulae[is_visible_mask].copy()


    # 5. CALCOLO PRIORITÀ OSSERVATIVA
    # Score basato su: magnitudine, altezza massima, separazione dalla Luna, massa d'aria
    visible_df['Priority_Score'] = 0

    # Normalizza i parametri per il calcolo del punteggio
    if not visible_df['V-Mag'].isna().all():
        # Invert magnitude scale (lower is brighter)
        mag_norm = (visible_df['V-Mag'].max() - visible_df['V-Mag'].fillna(visible_df['V-Mag'].max())) / (visible_df['V-Mag'].max() - visible_df['V-Mag'].min()) * 30 # Max 30 points
        visible_df['Priority_Score'] += mag_norm.fillna(0) # Handle NaNs after normalization

    alt_norm = visible_df['Max_Alt'] / 90 * 25  # Max 25 points
    visible_df['Priority_Score'] += alt_norm

    moon_norm = np.minimum(visible_df['Moon_Sep'] / 180 * 20, 20)  # Max 20 points (further is better)
    visible_df['Priority_Score'] += moon_norm

    # Airmass: lower is better. Airmass of 1 is at zenith (90 deg alt). Roughly Airmass ~ 1/cos(90-alt).
    # Good airmass is usually < 2 (alt > 30 deg).
    # Let's give higher score for lower airmass (higher altitude)
    # Max score at airmass 1, 0 score at airmass >= 3
    airmass_norm = np.maximum(0, (3 - visible_df['Min_Airmass']) / 2 * 15)  # Max 15 points
    visible_df['Priority_Score'] += airmass_norm

    # Bonus per oggetti grandi e luminosi
    size_bonus = np.where(visible_df['MajAx'] > 5, 10, 0)
    visible_df['Priority_Score'] += size_bonus


    # 6. STAMPA E SALVATAGGIO DEI RISULTATI
    if visible_df.empty:
        print("\n❌ Nessuna nebulosa trovata che soddisfi i criteri per questa sera.")
    else:
        print(f"\n🌟 TROVATE {len(visible_df)} NEBULOSE VISIBILI!")

        # Ordina per punteggio di priorità
        sorted_df = visible_df.sort_values(by='Priority_Score', ascending=False)

        # Statistiche per tipo
        print("\n📊 DISTRIBUZIONE PER TIPO:")
        type_counts = sorted_df['Type_Desc'].value_counts()
        for nebula_type, count in type_counts.items():
            print(f"   {nebula_type}: {count}")

        # Top consigliate
        print(f"\n🎯 TOP {TOP_Nebulose} NEBULOSE CONSIGLIATE (ordinate per priorità osservativa):")
        print("-" * 120)
        print("{:<12} {:<18} {:<8} {:<8} {:<10} {:<12} {:<8} {:<10} {:<12}".format(
            "Nome", "Tipo", "Costellaz", "Magn.", "Dim. (′)", "Ora ottimale", "Alt. Max", "Difficoltà", "Punteggio"
        ))
        print("-" * 120)

        for _, row in sorted_df.head(TOP_Nebulose).iterrows():
            mag_str = f"{row['V-Mag']:.1f}" if pd.notnull(row['V-Mag']) else "N/D"
            size_str = f"{row['MajAx']:.1f}" if pd.notnull(row['MajAx']) else "N/D"
            time_str = row['Optimal_Time'] if pd.notnull(row['Optimal_Time']) else "N/D" # Optimal_Time is already string

            print("{:<12} {:<18} {:<8} {:<8} {:<10} {:<12} {:<8}° {:<10} {:<12.1f}".format(
                str(row['Name'])[:12],
                str(row['Type_Desc'])[:18],
                str(row['Const'])[:8],
                mag_str,
                size_str,
                time_str,
                f"{row['Max_Alt']:.0f}",
                str(row['Difficulty'])[:10],
                row['Priority_Score']
            ))

        # Oggetti facili per principianti
        easy_objects = sorted_df[sorted_df['Difficulty'].str.contains('Facile', na=False)]
        if not easy_objects.empty:
            print("\n👶 OGGETTI FACILI PER PRINCIPIANTI:")
            print("-" * 80)
            for _, row in easy_objects.head(5).iterrows():
                mag_str = f"mag {row['V-Mag']:.1f}" if pd.notnull(row['V-Mag']) else ""
                size_str = f"{row['MajAx']:.0f}′" if pd.notnull(row['MajAx']) else ""
                print(f"   {row['Name']:<12} ({row['Type_Desc']}) in {row['Const']:<10} {mag_str:<10} {size_str}")

        # Avvisi per oggetti vicini alla Luna
        close_to_moon = sorted_df[sorted_df['Moon_Sep'] < 30]
        if not close_to_moon.empty:
            print("\n⚠️  ATTENZIONE - Oggetti vicini alla Luna (< 30°):")
            for _, row in close_to_moon.iterrows():
                print(f"   {row['Name']} - Separazione: {row['Moon_Sep']:.1f}°")

        # Salvataggio risultati
        try:
            # Prepara DataFrame per export
            export_columns = ['Name', 'Type_Desc', 'Const', 'V-Mag', 'MajAx', 'MinAx',
                            'Max_Alt', 'Optimal_Time', 'Min_Airmass', 'Surface_Brightness',
                            'Moon_Sep', 'Difficulty', 'Priority_Score']

            export_df = sorted_df[export_columns].copy()

            # Optimal_Time is already string, no need to reformat
            # export_df['Optimal_Time'] = export_df['Optimal_Time'].apply(
            #     lambda x: x.strftime('%H:%M') if pd.notnull(x) else 'N/D'
            # )

            filename = f"piano_osservazione_nebulose_{now.strftime('%Y%m%d')}.csv"
            export_df.to_csv(filename, index=False, sep=';', decimal=',')

            print("\n" + "=" * 80)
            print(f"💾 Piano completo salvato in: {filename}")
            print(f"   Contiene {len(export_df)} oggetti con tutti i parametri calcolati")

        except Exception as e:
            print(f"\n❌ Errore durante il salvataggio del file CSV: {e}")

        # Consigli finali
        print("\n" + "=" * 80)
        print("💡 CONSIGLI PER L'OSSERVAZIONE:")
        print("   • Inizia con gli oggetti a priorità più alta")
        print("   • Osserva gli oggetti quando sono vicini al meridiano (massima altezza)")
        print("   • Per nebulose ad emissione, considera l'uso di filtri OIII o H-alpha")
        print("   • La brillanza superficiale è più importante della magnitudine totale")
        print(f"   • Con massa d'aria > 2, aspettati un calo di trasparenza del {(2-1)*0.2*100:.0f}%")
        print("=" * 80)

# Esegui la funzione principale
if __name__ == "__main__":
    run_visibility_check()

🔭 PIANO DI OSSERVAZIONE NEBULOSE - 12/09/2025
📍 Località: Lat 41.7024°, Lon 15.7309°, Alt 566m
⏰ Finestra osservativa: 20:30 - 02:00

🌙 CONDIZIONI DEL CIELO:
   Altezza del Sole a metà serata: -39.3°
   Altezza della Luna: 15.5°
   Fase lunare: 33%
   Qualità del cielo: Buona (Primo quarto)

📂 Caricamento catalogo NGC...
✅ Trovate 260 nebulose nel catalogo dopo i filtri iniziali.
🔬 Calcolo visibilità e parametri osservativi...

🌟 TROVATE 104 NEBULOSE VISIBILI!

📊 DISTRIBUZIONE PER TIPO:
   Nebulosa: 37
   Emissione HII: 27
   Ammasso+Nebulosa: 18
   Planetaria: 15
   Resto Supernova: 7

🎯 TOP 20 NEBULOSE CONSIGLIATE (ordinate per priorità osservativa):
------------------------------------------------------------------------------------------------------------------------
Nome         Tipo               Costellaz Magn.    Dim. (′)   Ora ottimale Alt. Max Difficoltà Punteggio   
--------------------------------------------------------------------------------------------------------------

In [5]:
# Install astropy and other necessary libraries
!pip install --upgrade astropy pandas pytz numpy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 30.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 30.7 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.2
    Uninstalling pandas-2.2.2:
      Successfully uninstalled pandas-2.2.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.2 which is incompatible.
opencv-contrib-python 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 2.3.3 which i

In [5]:
# Install necessary libraries including astroplan
!pip install --upgrade astropy astroplan pandas pytz numpy